In [ ]:
from dataset_loader.dataset_loader import dataset_loader


_, _, test_data_p = dataset_loader('cora')

In [ ]:
from torch_geometric.datasets import Planetoid

dataset = Planetoid(root="dataset", name="Cora", split="full")
dataset = dataset[0]  # there is only one graph in the dataset

test_data = dataset.subgraph(dataset.test_mask)

for i in range(len(test_data.y)):
	print(test_data.y[i], test_data_p.y[i])

In [ ]:
from torch_geometric.datasets import Planetoid
import os
from torch_geometric.loader import DataLoader
import torch_geometric
import torch
import torch.nn.functional as F
from dataset_loader.dataset_loader import dataset_loader


DATASET_STORAGE_PATH = "./dataset/"

train_data, val_data, test_data = dataset_loader('cora')



In [ ]:
# count the how many elements are for each class (are 1-hot encoded)
print(test_data.y[:,0].sum())
print(test_data.y[:,1].sum())
print(test_data.y[:,2].sum())
print(test_data.y[:,3].sum())


In [ ]:
from utils.math import interval_softmax, actually_reachable
import torch
import numpy as np

a_L = np.array([9.6062404e-06, 9.9997389e-01, 1.6473108e-05])
a_U = np.array([9.6254726e-06, 9.9997395e-01, 1.6506086e-05])

assert np.all(a_L <= a_U), "Lower bound should be less than upper bound"

q_L, q_U = actually_reachable(a_L, a_U)

assert np.all(q_L <= q_U), "Lower bound should be less than upper bound"

print("Lower bound: ", q_L)
print("Upper bound: ", q_U)

In [ ]:
from dataset_loader.dataset_loader import dataset_loader

train_loader, val_loader, test_loader = dataset_loader('ogb-arxiv-year', {"batch_size":1})

In [ ]:
from dataset_loader.dataset_loader import dataset_loader

train_loader, val_loader, test_loader = dataset_loader('cora', {"batch_size":1})


In [ ]:
for data in test_loader:
	print(data.y[data.y.sum(dim=1) == 0].shape)
	break

In [ ]:
import gdown
dataset_drive_url = {
    # 'twitch-gamer_feat' : '1fA9VIIEI8N0L27MSQfcBzJgRQLvSbrvR',
    # 'twitch-gamer_edges' : '1XLETC6dG3lVl7kDmytEJ52hvDMVdxnZ0',
    'snap-patents' : '1ldh23TSY1PwXia6dU0MYcpyEgX-w3Hia', 
    'pokec' : '1dNs5E7BrWJbgcHeQ_zuy5Ozp2tRCWG0y', 
    'yelp-chi': '1fAXtTVQS4CfEk4asqrFw9EPmlUPGbGtJ', 
    'wiki_views': '1p5DlVHrnFgYm3VsNIzahSsvCD424AyvP', # Wiki 1.9M 
    'wiki_edges': '14X7FlkjrlUgmnsYtPwdh-gGuFla4yb5u', # Wiki 1.9M 
    'wiki_features': '1ySNspxbK-snNoAZM7oxiWGvOnTRdSyEK' # Wiki 1.9M
}

for key, value in dataset_drive_url.items():
	gdown.download(id=value, \
		output=f'{DATASET_STORAGE_PATH}/{key}.mat', quiet=False)

In [ ]:
for batch in train_loader:
	print(batch)
	print(batch.y[:,0].sum())
	print(batch.y[:,1].sum())
	print(batch.y[:,2].sum())
	print(len(batch.y[batch.y.sum(dim=1) == 0]))
	break

In [ ]:
stringa = "1fA9VIIEI8N0L27MSQfcBzJgRQLvSbrvR"

"1fA9VIIEI8N0L27MSQfcBzJgRQLvSbrvR" in stringa.lower()

In [ ]:
from torch_geometric.datasets import HeterophilousGraphDataset

dataset = HeterophilousGraphDataset(root="dataset", name="Roman-empire")

In [ ]:

dataset[0]

y = dataset[0].y

# check the values of y
print(y.unique())

# print how many elements are for each class
for i in range(y.max() + 1):
	print(f"Class {i}: {y[y == i].shape[0]} elements")


In [ ]:
import os
import os.path as osp
import ssl
import sys
import urllib
from typing import Optional

import fsspec

from torch_geometric.io import fs


def download_url(
    url: str,
    folder: str,
    log: bool = True,
    filename: Optional[str] = None,
):
    r"""Downloads the content of an URL to a specific folder.

    Args:
        url (str): The URL.
        folder (str): The folder.
        log (bool, optional): If :obj:`False`, will not print anything to the
            console. (default: :obj:`True`)
        filename (str, optional): The filename of the downloaded file. If set
            to :obj:`None`, will correspond to the filename given by the URL.
            (default: :obj:`None`)
    """
    if filename is None:
        filename = url.rpartition('/')[2]
        filename = filename if filename[0] == '?' else filename.split('?')[0]

    path = osp.join(folder, filename)

    if fs.exists(path):  # pragma: no cover
        if log and 'pytest' not in sys.modules:
            print(f'Using existing file {filename}', file=sys.stderr)
        return path

    if log and 'pytest' not in sys.modules:
        print(f'Downloading {url}', file=sys.stderr)

    os.makedirs(folder, exist_ok=True)

    context = ssl._create_unverified_context()
    data = urllib.request.urlopen(url, context=context)

    with fsspec.open(path, 'wb') as f:
        # workaround for https://bugs.python.org/issue42853
        while True:
            chunk = data.read(10 * 1024 * 1024)
            if not chunk:
                break
            f.write(chunk)

    return path


def download_google_url(
    id: str,
    folder: str,
    filename: str,
    log: bool = True,
):
    r"""Downloads the content of a Google Drive ID to a specific folder."""
    url = f'https://drive.usercontent.google.com/download?id={id}&confirm=t'
    return download_url(url, folder, log, filename)


In [ ]:
from typing import Callable, Optional, List
import os.path as osp
import json
import numpy as np
import torch
from torch_geometric.data import Data, InMemoryDataset

class Reddit2(InMemoryDataset):
    adj_full_id = '1sncK996BM5lpuDf75lDFqCiDZyErc1c2'
    feats_id = '1ZsHaJ0ussP1W722krmEIp_8pwKAoi5b3'
    class_map_id = '1JF3Pjv9OboMNYs2aXRQGbJbc4t_nDd5u'
    role_id = '1nJIKd77lcAGU4j-kVNx_AIGEkveIKz3A'

    def __init__(
        self, root: str, transform: Optional[Callable] = None,
        pre_transform: Optional[Callable] = None, force_reload: bool = False,
    ) -> None:
        super().__init__(root, transform, pre_transform, force_reload=force_reload)
        self.load(self.processed_paths[0])

    @property
    def raw_file_names(self) -> List[str]:
        return ['adj_full.npz', 'feats.npy', 'class_map.json', 'role.json']

    @property
    def processed_file_names(self) -> str:
        return 'data.pt'

    def download(self) -> None:
        download_google_url(self.adj_full_id, self.raw_dir, 'adj_full.npz')
        download_google_url(self.feats_id, self.raw_dir, 'feats.npy')
        download_google_url(self.class_map_id, self.raw_dir, 'class_map.json')
        download_google_url(self.role_id, self.raw_dir, 'role.json')

    def process(self) -> None:
        import scipy.sparse as sp
        f = np.load(osp.join(self.raw_dir, 'adj_full.npz'))
        adj = sp.csr_matrix((f['data'], f['indices'], f['indptr']), f['shape'])
        adj = adj.tocoo()
        row = torch.from_numpy(adj.row).to(torch.long)
        col = torch.from_numpy(adj.col).to(torch.long)
        edge_index = torch.stack([row, col], dim=0)
        x = torch.from_numpy(np.load(osp.join(self.raw_dir, 'feats.npy'))).to(torch.float)
        ys = [-1] * x.size(0)
        with open(osp.join(self.raw_dir, 'class_map.json')) as f:
            class_map = json.load(f)
            for key, item in class_map.items():
                ys[int(key)] = item
        y = torch.tensor(ys)
        with open(osp.join(self.raw_dir, 'role.json')) as f:
            role = json.load(f)
        train_mask = torch.zeros(x.size(0), dtype=torch.bool)
        train_mask[torch.tensor(role['tr'])] = True
        val_mask = torch.zeros(x.size(0), dtype=torch.bool)
        val_mask[torch.tensor(role['va'])] = True
        test_mask = torch.zeros(x.size(0), dtype=torch.bool)
        test_mask[torch.tensor(role['te'])] = True
        data = Data(x=x, edge_index=edge_index, y=y, train_mask=train_mask,
                    val_mask=val_mask, test_mask=test_mask)
        data = data if self.pre_transform is None else self.pre_transform(data)
        self.save([data], self.processed_paths[0])
# --- End of Provided Class ---


dataset = Reddit2(root='dataset')


In [ ]:
dataset[0]

In [ ]:
from dataset_loader.dataset_loader import dataset_loader

train_loader, val_loader, test_loader = dataset_loader('reddit2', {"batch_size":1})



In [ ]:
train_loader

In [ ]:
import numpy as np

path = "/vast/-/graph-uncertainty/dataset/roman_empire/roman_empire.npz"

# load the dataset

data = np.load(path)

In [ ]:
print(data.keys)


In [ ]:
# === Amazon Ratings Dataset Deep Summary ===
NPZ_PATH = "/vast/-/graph-uncertainty/dataset/roman_empire/roman_empire.npz"

import numpy as np
from collections import Counter

npz = np.load(NPZ_PATH, allow_pickle=True)
print("== Keys ==")
print(list(npz.keys()))

# --- Node info ---
x = npz["node_features"]
y = npz["node_labels"].reshape(-1)
edges = npz["edges"]

print(f"\n== Node features ==")
print(f"Shape: {x.shape}, dtype: {x.dtype}")
print(f"Example features (first row): {x[0][:10]}")

print(f"\n== Node labels ==")
unique, counts = np.unique(y, return_counts=True)
for u, c in zip(unique, counts):
    print(f"  Label {u}: {c}")
print(f"Num classes: {len(unique)}, total nodes: {len(y)}")

# --- Edge info ---
E = np.asarray(edges)
print(f"\n== Edges ==")
print(f"Shape: {E.shape}, dtype: {E.dtype}")
if E.shape[0] == 2:
    print("Edges stored as shape (2, E).")
elif E.shape[1] == 2:
    print("Edges stored as shape (E, 2).")
else:
    print("Unexpected edge shape!")
print(f"Min node index: {E.min()}, max node index: {E.max()}")

# --- Mask info ---
def summarize_mask(name, arr, expected_nodes):
    arr = np.asarray(arr)
    print(f"\n== {name} ==")
    print(f"Shape: {arr.shape}, dtype: {arr.dtype}")
    if arr.ndim == 2:
        print(f"Interpreting as (splits, nodes). #splits={arr.shape[0]}, #nodes={arr.shape[1]}")
        true_counts = arr.sum(axis=1)
        print(f"True counts per split (first 10): {true_counts[:10]}")
        if arr.shape[1] != expected_nodes:
            print(f"⚠️  WARNING: columns ({arr.shape[1]}) != num_nodes ({expected_nodes})")
    elif arr.ndim == 1:
        print(f"1D mask with {arr.sum()} True values out of {arr.size}")
        if arr.size != expected_nodes:
            print(f"⚠️  WARNING: length {arr.size} != num_nodes {expected_nodes}")

if "train_masks" in npz:
    summarize_mask("train_masks", npz["train_masks"], len(y))
if "val_masks" in npz:
    summarize_mask("val_masks", npz["val_masks"], len(y))
if "test_masks" in npz:
    summarize_mask("test_masks", npz["test_masks"], len(y))

print("\n✅ Done. Copy the full output here so I can confirm the structure precisely.")


In [ ]:
from dataset_loader.loader_roman_empire import load_roman_empire
from dataset_loader.loader_amazon_ratings import load_amazon_ratings
# cora
from dataset_loader.loader_cora import loader_cora

In [ ]:
load_roman_empire("./dataset/", config = {} )
load_amazon_ratings("./dataset/", config = {} )
loader_cora("./dataset/", config = {} )